# numpy — smoke test

The examples are **runtime** exercises, not import checks: each drives a real
compute path (BLAS `dgemm`, LAPACK `dgesdd`/`dsyevd`/`dgesv`/`dpotrf`, pocketfft,
SIMD-dispatched reductions) and asserts numeric correctness, so a broken BLAS/LAPACK
or FFT backend makes the notebook fail.

## Example 1 — BLAS GEMM (`dgemm`) on 500×500 float64

In [ ]:
import numpy as np
print("numpy", np.__version__)
rng = np.random.default_rng(0)
A = rng.standard_normal((500, 500))
B = rng.standard_normal((500, 500))
C = A @ B                                  # BLAS dgemm
assert C.shape == (500, 500) and C.dtype == np.float64 and np.isfinite(C).all()
assert np.allclose(A @ np.eye(500), A)     # multiplicative identity
ref = sum(A[3, k] * B[k, 7] for k in range(500))
assert np.isclose(C[3, 7], ref)            # element vs explicit dot product
assert np.allclose((A @ B).T, B.T @ A.T)   # (AB)^T == B^T A^T
print("GEMM 500x500 ok; C[0,0] =", C[0, 0])

## Example 2 — LAPACK SVD + symmetric eigendecomposition

In [ ]:
import numpy as np
rng = np.random.default_rng(1)
M = rng.standard_normal((200, 120))
U, S, Vh = np.linalg.svd(M, full_matrices=False)          # LAPACK dgesdd
assert np.all(S >= 0) and np.all(np.diff(S) <= 1e-9)      # non-negative, descending
assert np.allclose(U @ np.diag(S) @ Vh, M, atol=1e-8)     # reconstruction
Ssym = M.T @ M                                            # symmetric PSD
w, V = np.linalg.eigh(Ssym)                               # LAPACK dsyevd
assert np.all(w >= -1e-6)
assert np.allclose(V @ np.diag(w) @ V.T, Ssym, atol=1e-6) # reconstruction
assert np.isclose(S[0] ** 2, w[-1], rtol=1e-6)           # SVD <-> eigh cross-check
print("svd/eigh ok; top singular value =", S[0])

## Example 3 — `einsum` (matmul, trace, batched, covariance)

Exercises the einsum engine (with `optimize=True` path selection) and checks each
contraction against the equivalent explicit numpy op.

In [ ]:
import numpy as np
rng = np.random.default_rng(2)
X = rng.standard_normal((64, 32))
Y = rng.standard_normal((32, 16))
assert np.allclose(np.einsum("ij,jk->ik", X, Y, optimize=True), X @ Y)
Sq = rng.standard_normal((40, 40))
assert np.isclose(np.einsum("ii->", Sq), np.trace(Sq))          # trace
batch = rng.standard_normal((8, 5, 5))
assert np.allclose(np.einsum("bij,bjk->bik", batch, batch), batch @ batch)
cov = np.einsum("ni,nj->ij", X, X) / X.shape[0]                 # Gram / covariance
assert np.allclose(cov, cov.T)
print("einsum ok; cov shape =", cov.shape)

## Example 4 — FFT round-trip (pocketfft)

`rfft`/`irfft` and `fft2`/`ifft2` round-trips plus a spectral-peak check: a
50 Hz + 120 Hz signal must place its dominant `rfft` bin at 50 Hz.

In [ ]:
import numpy as np
rng = np.random.default_rng(3)
t = np.linspace(0, 1, 1024, endpoint=False)
sig = np.sin(2 * np.pi * 50 * t) + 0.5 * np.sin(2 * np.pi * 120 * t)
spec = np.fft.rfft(sig)
assert np.allclose(np.fft.irfft(spec, n=sig.size), sig, atol=1e-10)   # 1-D round-trip
freqs = np.fft.rfftfreq(sig.size, d=1 / 1024)
peak = freqs[np.argmax(np.abs(spec))]
assert abs(peak - 50) < 1.0                                          # dominant tone
img = rng.standard_normal((32, 48))
assert np.allclose(np.fft.ifft2(np.fft.fft2(img)).real, img, atol=1e-10)  # 2-D round-trip
print("fft ok; dominant peak =", peak, "Hz")

## Example 5 — solve / inv / Cholesky + strided SIMD reduction + Generator determinism

In [ ]:
import numpy as np
rng = np.random.default_rng(4)
n = 300
P = rng.standard_normal((n, n))
SPD = P @ P.T + n * np.eye(n)                              # symmetric positive-definite
b = rng.standard_normal(n)
x = np.linalg.solve(SPD, b)                                # LAPACK dgesv
assert np.allclose(SPD @ x, b, atol=1e-6)
assert np.allclose(SPD @ np.linalg.inv(SPD), np.eye(n), atol=1e-6)   # LAPACK gesv on the identity
L = np.linalg.cholesky(SPD)                                # dpotrf
assert np.allclose(L @ L.T, SPD, atol=1e-6)
big = np.arange(1_000_000, dtype=np.float64)
assert np.isclose(big[::2].sum(), sum(range(0, 1_000_000, 2)))       # strided SIMD reduction
r1 = np.random.default_rng(42).standard_normal(5)
r2 = np.random.default_rng(42).standard_normal(5)
assert np.array_equal(r1, r2)                              # PCG64 determinism
print("solve/inv/cholesky/strided/rng ok")